In [ ]:
# =====================================================================
# STEP 1: CLEAN UP & PREPARATION
# =====================================================================
# Install zstd (required by Ollama for extraction)
!sudo apt-get update && sudo apt-get install -y zstd

# FORCE-KILL any existing or frozen Ollama servers running in Colab
!pkill -9 -f ollama || true

# =====================================================================
# STEP 2: INSTALL OLLAMA & NGROK
# =====================================================================
# Install Ollama in the Colab container
!curl -fsSL https://ollama.com/install.sh | sh

# FORCE-KILL again immediately after install to clear port 11434
!pkill -9 -f ollama || true

# Install pyngrok to handle the secure tunnel
!pip install pyngrok

# =====================================================================
# STEP 3: AUTHENTICATE NGROK
# =====================================================================
# Paste your Ngrok Auth Token inside the quotes below:
from pyngrok import ngrok
ngrok.set_auth_token("ngrock key")

# =====================================================================
# STEP 4: START OLLAMA WITH EXTERNAL ACCESS
# =====================================================================
import subprocess
import time
import os

# Configure Ollama to bind to all interfaces & accept external origins
os.environ["OLLAMA_ORIGINS"] = "*"  
os.environ["OLLAMA_HOST"] = "0.0.0.0"

print("\nStarting fresh Ollama server in background...")
# Redirect logs to a physical file (ollama.log) to avoid Jupyter environment fileno errors
log_file = open("ollama.log", "w")
subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)
time.sleep(6)  # Wait for it to boot

# =====================================================================
# STEP 5: DOWNLOAD THE ADVANCED AI MODEL
# =====================================================================
# Downloads Gemma 2 (9B, 5.5GB). Fits beautifully in Colab's 16GB VRAM!
print("\nDownloading Gemma 2 model...")
!ollama pull gemma2

# =====================================================================
# STEP 6: RUN STARTUP SELF-TEST
# =====================================================================
print("\n--- RUNNING LOCAL CONNECTION SELF-TEST ---")
!curl -i -H "Origin: http://external-pc-test.com" http://localhost:11434/api/tags
print("-------------------------------------------\n")

# =====================================================================
# STEP 7: EXPOSE OLLAMA TO THE INTERNET WITH HOST REWRITING
# =====================================================================
# Disconnect any old active tunnels first
ngrok.kill()

# We pass host_header="rewrite" to automatically trick Ollama into accepting the requests!
public_url = ngrok.connect(11434, "http", host_header="rewrite")

print("\n🎉 CONNECTION SUCCESSFUL! 🎉")
print(f"Copy this link and paste it into bone_Brain.py as OLLAMA_HOST:")
print(f"👉 OLLAMA_HOST = \"{public_url.public_url}\"")
print("\nOllama is now live and waiting for requests! Keep this Colab cell running.")
